## Agenda

#### 1 Create spark session

  - installing some libraries
  - Restarting python (in case previous libraries installation asks to do so)
  - Importing some libraries

#### 2 create dataframe (by reading a file (csv, json, etc.))

- 1 By reading a file in the Volume(a csv, json, etc.) FOOL(Format, Option, Option, Load)
    -  using  spark.read.format().option().option().load()
- 2 By reading a table where the data sits using spark.read.table
- 3 Check the notebook "3_Customer_Dataframe_Creation".  using spark.createDataFrame(data=data_list, schema=data_schema)

#### 3 Query the data the SQL Query way and the PySpark Transformation way
  -   SQL Query way
  -   PySpark transformations way


#### 4 Example

--------------------------------------

##### Step One: Creating spark session

 spark.version

##### Step Two: Creating datframe
First of all you must create a dataframe and here we list 3 different ways, although we will describe 2 in this notebook, the 3rd one is discussed in a previous nortebook.

 1 create a Dataframe by reading a file in the Volume(a csv, json, etc.)

-     using spark.read.format
                      .option
                      .option
                      .load

2 Create a dataframe by reading a table where the data sits

-     using spark.read.table

3 Check the notebook "3_Customer_Dataframe_Creation". We are not reviewing this one for this practice

-     using spark.createDataFrame(data=data_list, schema=data_schema)


##### Step Three: Query the data SQL Query way and PySpark Transformations way
------------------------------------------------------------------------------------------
###### SQL Query way --> Write a normal SQL query 

###### PySpark transformations way --> PySpark divides into several steps to perform the steps of an SQL query

 -    < 1 Read teh data

 -    < 2 Apply transformations(querying the data)
        - IDEAL :    ENCAPSULATES ALL TRANSFORMATIONS INTO ONE dataframe
        - NOT IDEAL: Creates a dataframe per transformation

 -    < 3 Show/Execute the result/Actions(applied to teh result)

####5 Practices

![image_1774032783996.png](./image_1774032783996.png "image_1774032783996.png")


### 4 Example

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 1 Create spark session


In [0]:
#This below is a pre-created spark session. It's used when you want t avoid creating the spark session with teh builder. 
# We will use this to create a spark session from here onwards
#spark.version

##### 1.1 Installing some libraries

In [0]:
#pip install duckdb pandas # install duckdb and pandas to be able to query a dataframe

##### 1.2 Restarting python

In [0]:
#dbutils.library.restartPython() # tHIS restarts the kernel or python after running the above command to install duckdb

##### 1.3 Importing some libraries

In [0]:
"""
import pandas as pd
import duckdb
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import to_date, col
"""

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 2 Create dataframe (by reading a file from the Volume(csv, json, etc.))

###### Choose one these three below. 



Before starting you can take different approaches about creating the dataframes, you can:

 <> create a Dataframe by reading a file (in the Volume a csv, json, etc.) 
-     using spark.read.format
                      .option
                      .option
                      .load         

 <> Create a dataframe by reading a table where the data sits  
-     using spark.read.table

 <> Check the notebook "3_Customer_Dataframe_Creation". We are not reviewing this one for this practice
    
-     df_raw = spark.createDataFrame(data=data_list, schema=data_schema)




In [0]:
# Examples of How to read a csv and a json file
#  
# Reading a csv file
#file_df = ( spark.read.format('csv')
#                      .option('header', 'true')
#                      .option('inferSchema', 'true')
#                      .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
#          )

#Read a json file
#Using a connector(options).
#json_file_df = (
#                spark.read.format('json')
#                .load(path= '/Volumes/dev/spark_db/datasets/spark_programming/data/diamonds.json')
#              )


-----------------------------------------------------------------------------
##### <> using spark.read.format

In [0]:
# Reading a csv file
#df = ( spark.read.format('csv')
#                 .option('header', 'true')
#                 .option('inferSchema', 'true')
#                 .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/customers.csv")
#     )

-----------------------------------------------------------------------------
##### <>  using spark.read.table()

In [0]:
# 1 Read data, table in this case
#cust_df =  spark.read.table("dev.spark_db.customers")

###### Let's choose teh first method

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 3 Query the data SQL Query way and PySpark Transformations way

###### Before anything you MUST create a dataframe




-----------------------------------------------------------------------------
-----------------------------------------------------------------------------
#### Request: Query the Top three selling products

##### SQL Query 

In [0]:
%sql

-- SELECT PRODUCTID
-- FROM dev.spark.db.customers
-- ORDER BY quantity desc 
-- LIMIT 3;

-----------------------------------------------------------------------------

##### PySpark  divides in 3 steps to perform the above query

< 1/3 Read teh data
  
-     Creates a df with one of the methods mentioned above

< 2/3 Apply transformations(querying the data)

-     Replicate the query the PySpark's way
      - Ideal way (ONE DF <all steps encapsulated>)
      - Not Ideal way (SEVRAL DFs <all steps divided>)

< 3/3 Show/Execute the result/Actions(applied to teh result)


-----------------------------------------------------------------------------
##### <1/3 Read teh data. A table in this case

In [0]:
# 1 Read data, table in this case
#cust_df =  spark.read.table("dev.spark_db.customers")

------------------------------------------------------------------------------
IDEAL(ONE DF (all steps encapsulated))
##### <2/3 Apply transformations(Composable query: All steps in encapsualted) 
All transformations are done in one encapsulated query and saved to a new dataframe instead of dividing in datframes)

In [0]:
"""
from pyspark.sql.functions import expr

result_df = ( cust_df.select("customerid", "customer_name", "productid", "quantity", "price")
                     .where("productid IS NOT NULL")
                     .groupBy("customerid", "customer_name").agg(expr("sum(quantity * price)").alias("sales"))
                     .orderBy("sales", ascending=False)
                     .limit(3)
            )

result_df.display()        

"""

---------------------------------------------------------------------
NOT IDEAL (SEVERAL DFs (all steps divided))
##### <2/3 Apply transformations(Divide into steps). This is other way but reqires the creation of too many dataframes

Dividing the query in STEPS, ONE dataframe for EACH transformation


In [0]:
# The aggreations are not done in the SELECT , they are done in the GROUPBY
# expr() operates on a single SQL expression and returns a Column object for use within other DataFrame methods.
#  This function is useful when you want to apply a SINGLE SQL expression to a DataFrame column or perform a SQL-like operation on a DataFrame

# select  Returns a DataFrame with subset (or all) of columns.

In [0]:
#from pyspark.sql.functions import expr

#df1 = cust_df.select("customerid", "customer_name", "productid", "quantity", "price")
#df2 = df1.where("productid IS NOT NULL")
#df3 = df2.groupBy("customerid", "customer_name").agg(expr("sum(quantity * price)").alias("sales"))
#df4= df3.orderBy("sales", ascending=False)
#df5= df4.limit(3)
#df5.display()


![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")
#### 5 PRACTICES 

###### Start spark session

In [0]:
###### Start spark session
spark.version


'4.1.0'

###### Pip install libraries

In [0]:
pip install duckdb pandas # install duckdb and pandas to be able to query a dataframe

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


###### Restart python to catch libraries updates

In [0]:

dbutils.library.restartPython() #  Restarts the kernel or python after running the above command to install duckdb

###### Import libraries

In [0]:
###### Import libraries
import pandas as pd
import duckdb
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import to_date, col


--------------------------------------------------
###### SQL query request: Count Disctinct Productids
--------------------------------------------------

###### SQL query
----------------------------------------------------------------------

In [0]:
%sql
SELECT COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT 
FROM dev.spark_db.customers

DISTINCT_PRODUCT_COUNT
4


###### PySpark divides in three steps to perform the above query
----------------------------------------------------------------------

###### Reading csv file to create DataFrame

In [0]:
# Create dataframe by reading a csv file
df = ( spark.read.format('csv')
                 .option('header', 'true')
                 .option('inferSchema', 'true')
                 .load(path="/Volumes/dev/spark_db/datasets/spark_programming/data/customers.csv")
     )

###### Pyspark transformation of the SQL query
######Take the same piece of the SQL query above: COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT 

######and, in pyspark, put into the selectExpr() :  --> selectExpr('COUNT(DISTINCT PRODUCTID) AS DISTINCT_PRODUCT_COUNT ')


 

In [0]:
res_df2 = (df.selectExpr('count(distinct productid) as distinct_product_count') 
              )
res_df2.display()

distinct_product_count
4


----------------------------------------------------------------------
###### Request: Distinct customers 
----------------------------------------------------------------------


###### SQL query

In [0]:
%sql
select distinct customerid
from dev.spark_db.customers

customerid
1
2
3
4
5
6


###### PySpark divides in three steps to perform the above query
----------------------------------------------------------------------

###### Take the same piece of the SQL query above: distinct customerid 

###### and, in pyspark, put into the selectExpr() :  --> selectExpr('distinct customerid')

In [0]:
res_df3 = (df.select('customerid').distinct()
                )
res_df3.display()

customerid
5
1
3
2
6
4


![image_1774032783996.png](./image_1774032783996.png "image_1774032783996.png")


## THIS ONE BELOW IS DONE BY USING DuckDb but do not run it. Check later on

In [0]:
# 1 Read data, table in this case
df2 =  spark.read.table("dev.spark_db.customers")

In [0]:
#import pandas as pd
#import duckdb

#duckdb.register('df2', df2.toPandas())
#query = """
#SELECT distinct customerid
#FROM df2
#"""
#result = duckdb.query(query).df()
#print(result)

3 Find duplicates

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")

![image_1774032826788.png](./image_1774032826788.png "image_1774032826788.png")